<a href="https://colab.research.google.com/github/Sunn2x333/scalar_framework/blob/main/BBNv4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =============================================================================
# BBN_SFIB_v4 – Near-Transparency Demonstration with Strict Sign Tracking
# Author: Bobby Davis
# Verified: Multi-model review (June 2026)
# Changes from v4:
#   - Removed abs() masking of X_avg sign
#   - Updated Yp sensitivity to literature value 1.3e-4 s^-1
#   - Corrected phi_A units documentation to include *s factor
#   - Retained paper's (1 + k_phi*X) convention
#   - Retained bottle=spacelike, beam=baseline labeling
# =============================================================================

import numpy as np

c      = 2.99792458e8
hbar   = 1.05571817e-34
eV     = 1.60217662e-19
MeV    = 1e6 * eV

m_phi_eV = 1e-3            # field mass upper bound [eV] — fifth-force limit
k_phi    = 9.83e-11        # coupling constant [m^4/J] — lab calibration
g_star   = 10.75

def hubble_BBN(T_MeV):
    """Hubble rate in radiation domination [eV]."""
    return T_MeV**2 * np.sqrt(np.pi**2 * g_star / 90) / 2.435e18 * 1e6

print("="*70)
print("BBN_SFIB_v5 — Near-Transparency with Sign Tracking")
print("="*70)

# --- 1. DYNAMICAL REGIME ---
print("\n1. FIELD DYNAMICS ACROSS BBN EPOCH")
print(f"   m_phi = {m_phi_eV:.1e} eV  (fifth-force upper bound)")
print()
print(f"   {'T (MeV)':<10} {'H (eV)':<15} {'m_phi/H':<15} {'Regime'}")
print("   " + "-"*55)
for T in [3.0, 1.0, 0.8, 0.3, 0.1]:
    H = hubble_BBN(T)
    r = m_phi_eV / H
    print(f"   {T:<10.1f} {H:<15.3e} {r:<15.2e} {'Oscillating (m >> H)'}")

period_s = (2 * np.pi * hbar) / (m_phi_eV * eV)
n_osc    = 180.0 / period_s
print(f"\n   Oscillation period : {period_s:.3e} s")
print(f"   BBN duration       : 180 s (~3 minutes)")
print(f"   Cycles during BBN  : {n_osc:.3e}")
print(f"   → No spatial boundaries in homogeneous early universe → X purely timelike")

# --- 2. ENERGY BUDGET & SIGN-TRACKED SOURCE TERM ---
T_freeze    = 0.8
rho_rad     = (np.pi**2 / 30) * g_star * (T_freeze * MeV)**4 / (hbar * c)**3
rho_phi_max = 0.10 * rho_rad   # ΔNeff bound: rho_phi < 10% rho_rad

omega       = m_phi_eV * eV / hbar
phi_A_max   = np.sqrt(rho_phi_max) / omega  # canonical units: J^(1/2) m^(-3/2) s

# Homogeneous universe: grad(phi) = 0
# X(t) = -(phi_dot/c)^2 = -(phi_A omega/c)^2 sin^2(omega t)
# Time-averaged over ~10^13 oscillations: <sin^2> = 1/2
X_avg = -0.5 * (phi_A_max * omega / c)**2   # strictly negative (timelike)

# Native master equation: lambda = lambda_0 * (1 + k_phi * X)
# X_avg < 0 → delta_lambda < 0 (suppression)
delta_lambda_frac = k_phi * X_avg           # negative
delta_tau_frac    = -delta_lambda_frac      # positive (longer lifetime)

print(f"\n2. ENERGY BUDGET CONSTRAINT AT FREEZE-OUT (T = {T_freeze} MeV)")
print(f"   rho_rad                 : {rho_rad:.3e} J/m^3")
print(f"   Max rho_phi (10% budget): {rho_phi_max:.3e} J/m^3")
print(f"   phi_A upper bound       : {phi_A_max:.3e} J^(1/2) m^(-3/2) s")
print(f"   <X> time-averaged       : {X_avg:.4e} J/m^3  (strictly timelike, negative)")
print(f"   delta_lambda / lambda   : {delta_lambda_frac*100:.5f}%  (suppression)")
print(f"   delta_tau / tau         : +{delta_tau_frac*100:.5f}%  (longer tau_n)")

# --- 3. Yp IMPACT ---
# Literature sensitivity: dYp/d(tau_n) ~ 1.3e-4 per second
# (Steigman 2012; Iocco et al. 2009)
tau_n       = 879.4
delta_tau_s = delta_tau_frac * tau_n        # seconds
delta_Yp    = 1.3e-4 * delta_tau_s         # absolute Yp change

Yp_std  = 0.2470
Yp_obs  = 0.2450
sigma   = 0.0040
Yp_sfib = Yp_std + delta_Yp

print(f"\n3. HELIUM-4 ABUNDANCE")
print(f"   Standard BBN Yp         : {Yp_std:.4f}")
print(f"   delta_tau               : +{delta_tau_s:.4f} s")
print(f"   delta_Yp                : +{delta_Yp:.3e}  (mild upward nudge)")
print(f"   Yp with SFIB            : {Yp_sfib:.6f}")
print(f"   Observed Yp             : {Yp_obs:.4f} ± {sigma}")
print(f"   |delta_Yp| / sigma      : {abs(delta_Yp)/sigma:.3f}  (~{sigma/abs(delta_Yp) if delta_Yp != 0 else float("inf"):.0f}x below detection)")

# --- 4. NEUTRON LIFETIME CONSISTENCY ---
tau_BBN   = 870.0
sigma_BBN = 16.0

print(f"\n4. NEUTRON LIFETIME CONSISTENCY")
print(f"   tau_n bottle (lab)      : {tau_n} s  [bottle wall + B-field → spacelike X>0]")
print(f"   tau_n beam (lab)        : 888.0 s   [no confinement boundary → X≈0, baseline]")
print(f"   tau_n BBN+CMB           : {tau_BBN} ± {sigma_BBN} s  [Coc et al. 2014]")
print(f"   Bottle vs BBN           : {abs(tau_n-tau_BBN)/sigma_BBN:.1f}σ — consistent")

# --- 5. SIGN FLOW ---
print(f"\n5. SIGN FLOW  (master eq: lambda = lambda_0 [1 + k_phi * X])")
print(f"   Bottle (spacelike X>0) → lambda > lambda_0 → tau_bottle < tau_beam  ✓")
print(f"   BBN    (timelike X<0)  → lambda < lambda_0 → tau_n longer → Yp up   ✓")
print(f"   |delta_lambda/lambda| at BBN : {abs(delta_lambda_frac)*100:.4f}%")

print(f"\n{'='*70}")
print("CONCLUSION")
print(f"{'='*70}")
print(f"""
At BBN (T = 0.1–3 MeV):
  m_phi/H ~ 10^8–10^11  →  field oscillates ~10^13 times during nucleosynthesis
  No spatial boundaries in homogeneous universe  →  X purely timelike (negative)
  ΔNeff energy budget caps |delta_lambda/lambda| < {abs(delta_lambda_frac)*100:.2f}% at freeze-out
  |delta_Yp| = {abs(delta_Yp):.1e}  →  ~{sigma/abs(delta_Yp):.0f}x below astrophysical measurement floor

Self-consistency result: the same parameter constraints that satisfy fifth-force
bounds automatically ensure SFIB does not disrupt BBN. Standard abundances are
recovered to well within observational precision.

Sign flow confirms internal consistency across regimes:
  spacelike X > 0  (bottle, lab)   →  enhanced decay  →  explains beam/bottle gap
  timelike  X < 0  (BBN, cosmic)   →  suppressed decay →  Yp nudges up, undetectably

Li-7 overproduction (factor ~3.1x) is an open problem in standard BBN physics,
predating and independent of SFIB.
""")

BBN_SFIB_v5 — Near-Transparency with Sign Tracking

1. FIELD DYNAMICS ACROSS BBN EPOCH
   m_phi = 1.0e-03 eV  (fifth-force upper bound)

   T (MeV)    H (eV)          m_phi/H         Regime
   -------------------------------------------------------
   3.0        4.013e-12       2.49e+08        Oscillating (m >> H)
   1.0        4.459e-13       2.24e+09        Oscillating (m >> H)
   0.8        2.854e-13       3.50e+09        Oscillating (m >> H)
   0.3        4.013e-14       2.49e+10        Oscillating (m >> H)
   0.1        4.459e-15       2.24e+11        Oscillating (m >> H)

   Oscillation period : 4.140e-12 s
   BBN duration       : 180 s (~3 minutes)
   Cycles during BBN  : 4.348e+13
   → No spatial boundaries in homogeneous early universe → X purely timelike

2. ENERGY BUDGET CONSTRAINT AT FREEZE-OUT (T = 0.8 MeV)
   rho_rad                 : 3.011e+25 J/m^3
   Max rho_phi (10% budget): 3.011e+24 J/m^3
   phi_A upper bound       : 1.143e+00 J^(1/2) m^(-3/2) s
   <X> time-averaged